In [7]:
import sys
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display


# ============================================================
# Projektpfad und eigene Utilities
# ============================================================

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.saving_utils import DatasetBundle


# ============================================================
# Vollständige Discovery- und Validation-Datensätze laden
# ============================================================

data_discov_full = DatasetBundle.load(
    project_root / "data" / "preprocessed" / "data_discov_full"
)

data_valid_full = DatasetBundle.load(
    project_root / "data" / "preprocessed" / "data_valid_full"
)

data_discov_full.meta = cast(
    pd.DataFrame,
    data_discov_full.meta
)

data_valid_full.meta = cast(
    pd.DataFrame,
    data_valid_full.meta
)


# ============================================================
# Clusterlabels und Posterior-Wahrscheinlichkeiten laden
# ============================================================

proba_and_labels_discov = pd.read_parquet(
    project_root
    / "outputs"
    / "gmm_clustering"
    / "cluster_pred_discov.parquet"
)

proba_and_labels_valid = pd.read_parquet(
    project_root
    / "outputs"
    / "gmm_clustering"
    / "cluster_pred_valid.parquet"
)

proba_discov = proba_and_labels_discov.drop(
    columns="label"
)

proba_valid = proba_and_labels_valid.drop(
    columns="label"
)

labels_discov = proba_and_labels_discov["label"]
labels_valid = proba_and_labels_valid["label"]


# ============================================================
# Surveygewichte
# ============================================================

sample_weight_discov = (
    pd.to_numeric(
        data_discov_full.data["wichtn"],
        errors="coerce"
    )
    .to_numpy(dtype=float)
)

sample_weight_valid = (
    pd.to_numeric(
        data_valid_full.data["wichtn"],
        errors="coerce"
    )
    .to_numpy(dtype=float)
)


# ============================================================
# Gespeicherte Cluster-Summaries laden
# ============================================================

summaries_folder = (
    project_root
    / "outputs"
    / "gmm_clustering"
)

cluster_discov_summaries = pd.read_parquet(
    summaries_folder
    / "cluster_discov_summaries_not_nominal.parquet"
)

cluster_valid_summaries = pd.read_parquet(
    summaries_folder
    / "cluster_valid_summaries_not_nominal.parquet"
)


# ============================================================
# Cluster-ID als Index setzen
# ============================================================

cluster_discov_summaries.index = pd.Index(
    range(len(cluster_discov_summaries)),
    name="cluster"
)

cluster_valid_summaries.index = pd.Index(
    range(len(cluster_valid_summaries)),
    name="cluster"
)


# ============================================================
# Plausibilitätsprüfungen
# ============================================================

if len(labels_discov) != len(data_discov_full.data):
    raise ValueError(
        "Discovery-Labels und Discovery-Daten haben "
        "unterschiedliche Längen."
    )

if len(labels_valid) != len(data_valid_full.data):
    raise ValueError(
        "Validation-Labels und Validation-Daten haben "
        "unterschiedliche Längen."
    )

print(
    "Discovery:",
    len(data_discov_full.data),
    "Personen,",
    labels_discov.nunique(),
    "Cluster"
)

print(
    "Validation:",
    len(data_valid_full.data),
    "Personen,",
    labels_valid.nunique(),
    "Cluster"
)

print(
    "Discovery-Summaries:",
    cluster_discov_summaries.shape
)

print(
    "Validation-Summaries:",
    cluster_valid_summaries.shape
)

Loaded dataset from /Users/aginatore/Desktop/Consulting/esa_consulting/data/preprocessed/data_discov_full.
Loaded dataset from /Users/aginatore/Desktop/Consulting/esa_consulting/data/preprocessed/data_valid_full.
Discovery: 8835 Personen, 23 Cluster
Validation: 8588 Personen, 23 Cluster
Discovery-Summaries: (24, 214)
Validation-Summaries: (24, 214)


In [13]:
data_discov_full = DatasetBundle.load(
    project_root / "data/preprocessed/data_discov_full"
)

data_valid_full = DatasetBundle.load(
    project_root / "data/preprocessed/data_valid_full"
)

data_discov_full.meta = cast(
    pd.DataFrame,
    data_discov_full.meta
)

data_valid_full.meta = cast(
    pd.DataFrame,
    data_valid_full.meta
)

Loaded dataset from /Users/aginatore/Desktop/Consulting/esa_consulting/data/preprocessed/data_discov_full.
Loaded dataset from /Users/aginatore/Desktop/Consulting/esa_consulting/data/preprocessed/data_valid_full.


In [14]:
# ============================================================
# 1. Kontinuierliche und ordinale Variablen auswählen
# ============================================================

profile_variables = (
    data_discov_full.meta
    .loc[
        lambda df: df["type"].isin(["continuous", "ordinal"]),
        "name"
    ]
    .tolist()
)

print(
    f"Laut Metadaten ausgewählt: "
    f"{len(profile_variables)} kontinuierliche/ordinale Variablen"
)

import ast


# ============================================================
# 2. Gewichtete Clustermittelwerte aus den Summaries holen
# ============================================================

def extract_mean_columns(summary):
    """
    Extrahiert Mittelwertspalten aus einer Cluster-Summary.

    Erwartet Spaltennamen wie:
    "('alter', 'mean')"
    "('alter', 'std')"
    """
    summary = summary.copy()

    parsed_columns = {}

    for column in summary.columns:
        try:
            variable, statistic = ast.literal_eval(str(column))
        except (ValueError, SyntaxError, TypeError):
            continue

        if statistic == "mean":
            parsed_columns[column] = variable

    if not parsed_columns:
        raise ValueError(
            "In der Cluster-Summary wurden keine "
            "Mittelwertspalten gefunden."
        )

    result = summary.loc[:, list(parsed_columns)].copy()
    result = result.rename(columns=parsed_columns)

    return result


disc_cluster_means_all = extract_mean_columns(
    cluster_discov_summaries
)

valid_cluster_means_all = extract_mean_columns(
    cluster_valid_summaries
)

# Numerisch konvertieren
disc_cluster_means_all = disc_cluster_means_all.apply(
    pd.to_numeric,
    errors="coerce"
)

valid_cluster_means_all = valid_cluster_means_all.apply(
    pd.to_numeric,
    errors="coerce"
)

# Gewichtungsvariable entfernen
disc_cluster_means_all = disc_cluster_means_all.drop(
    columns=["wichtn"],
    errors="ignore"
)

valid_cluster_means_all = valid_cluster_means_all.drop(
    columns=["wichtn"],
    errors="ignore"
)

print(
    "Gefundene Mittelwertspalten:",
    f"Discovery={disc_cluster_means_all.shape[1]},",
    f"Validation={valid_cluster_means_all.shape[1]}"
)

print("\nErste Mittelwertspalten:")
print(disc_cluster_means_all.columns[:20].tolist())
# ============================================================
# 3. Gemeinsame Profilvariablen bestimmen
# ============================================================

common_variables = [
    variable
    for variable in profile_variables
    if variable in disc_cluster_means_all.columns
    and variable in valid_cluster_means_all.columns
    and variable in data_discov_full.data.columns
]

disc_cluster_means_all = (
    disc_cluster_means_all[common_variables]
    .copy()
)

valid_cluster_means_all = (
    valid_cluster_means_all[common_variables]
    .copy()
)

print(
    f"In beiden Cluster-Summaries vorhanden: "
    f"{len(common_variables)} Variablen"
)


# ============================================================
# 4. Discovery-Personendaten vorbereiten
# ============================================================

def series_to_numeric(series):
    """
    Konvertiert numerische und kategoriale Variablen robust
    in numerische Werte.

    Bei pandas-Categorical-Variablen werden zunächst die
    dargestellten Kategorien verwendet. Falls diese nicht
    numerisch sind, werden die geordneten Kategoriecodes genutzt.
    """
    if isinstance(series.dtype, pd.CategoricalDtype):
        category_values = pd.to_numeric(
            series.astype(str),
            errors="coerce"
        )

        # Numerisch beschriftete Kategorien, z. B. "1", "2", "3"
        if category_values.notna().any():
            return category_values

        # Nicht numerisch beschriftete ordinale Kategorien
        if series.cat.ordered:
            codes = series.cat.codes.astype(float)
            return codes.where(codes >= 0, np.nan)

        return pd.Series(np.nan, index=series.index, dtype=float)

    return pd.to_numeric(series, errors="coerce")


disc_person_data = pd.DataFrame(
    {
        variable: series_to_numeric(
            data_discov_full.data[variable]
        )
        for variable in common_variables
    },
    index=data_discov_full.data.index
)

disc_weights = np.asarray(
    sample_weight_discov,
    dtype=float
)

if len(disc_person_data) != len(disc_weights):
    raise ValueError(
        "Discovery-Personendaten und Gewichte haben "
        "unterschiedliche Längen."
    )


# ============================================================
# 5. Gewichteten Discovery-Mittelwert und gewichtete SD
#    für jede Variable berechnen
# ============================================================

def weighted_mean_sd_safe(data, weights):
    """
    Berechnet pro Variable den gewichteten Mittelwert und
    die gewichtete Populationsstandardabweichung.

    Fehlende Werte sowie nicht positive oder ungültige
    Gewichte werden ausgeschlossen.
    """
    weights = np.asarray(weights, dtype=float)

    means = {}
    standard_deviations = {}
    excluded = {}

    for variable in data.columns:
        values = pd.to_numeric(
            data[variable],
            errors="coerce"
        ).to_numpy(dtype=float)

        valid = (
            np.isfinite(values)
            & np.isfinite(weights)
            & (weights > 0)
        )

        x = values[valid]
        w = weights[valid]

        if len(x) == 0 or w.sum() <= 0:
            means[variable] = np.nan
            standard_deviations[variable] = np.nan
            excluded[variable] = (
                "keine gültigen Werte mit positivem Gewicht"
            )
            continue

        mean = np.average(
            x,
            weights=w
        )

        variance = np.average(
            (x - mean) ** 2,
            weights=w
        )

        sd = np.sqrt(variance)

        means[variable] = mean
        standard_deviations[variable] = sd

        if not np.isfinite(sd):
            excluded[variable] = "SD nicht definiert"
        elif sd == 0:
            excluded[variable] = "keine Varianz"

    return (
        pd.Series(means, dtype=float),
        pd.Series(standard_deviations, dtype=float),
        excluded
    )


discovery_mean, discovery_sd, excluded_variables = (
    weighted_mean_sd_safe(
        disc_person_data,
        disc_weights
    )
)

print("\nVariablen ohne verwendbare Discovery-SD:")

if excluded_variables:
    display(
        pd.Series(
            excluded_variables,
            name="Grund"
        ).to_frame()
    )
else:
    print("Keine")


# ============================================================
# 6. Nur Variablen mit gültiger positiver Discovery-SD behalten
# ============================================================

usable_variables = [
    variable
    for variable in common_variables
    if pd.notna(discovery_mean[variable])
    and pd.notna(discovery_sd[variable])
    and np.isfinite(discovery_sd[variable])
    and discovery_sd[variable] > 0
]

print(f"\nVerwendete Variablen: {len(usable_variables)}")
print(
    f"Entfernte Variablen: "
    f"{len(common_variables) - len(usable_variables)}"
)

if len(usable_variables) < 3:
    raise ValueError(
        "Für den Profilvergleich sind weniger als drei "
        "verwendbare Variablen vorhanden."
    )


# ============================================================
# 7. Clustergrößen bestimmen
# ============================================================

labels_discov_series = (
    pd.Series(labels_discov)
    .reset_index(drop=True)
)

labels_valid_series = (
    pd.Series(labels_valid)
    .reset_index(drop=True)
)

disc_cluster_counts = (
    labels_discov_series
    .value_counts()
    .sort_index()
)

valid_cluster_counts = (
    labels_valid_series
    .value_counts()
    .sort_index()
)

min_cluster_size = 100

large_disc_clusters = (
    disc_cluster_counts
    .loc[lambda counts: counts >= min_cluster_size]
    .index
    .tolist()
)

large_valid_clusters = (
    valid_cluster_counts
    .loc[lambda counts: counts >= min_cluster_size]
    .index
    .tolist()
)

print("\nClustergrößen Discovery:")
display(
    disc_cluster_counts
    .rename("n")
    .to_frame()
)

print("Clustergrößen Validation:")
display(
    valid_cluster_counts
    .rename("n")
    .to_frame()
)

print(
    f"Discovery-Cluster mit mindestens "
    f"{min_cluster_size} Personen:"
)
print(large_disc_clusters)

print(
    f"Validation-Cluster mit mindestens "
    f"{min_cluster_size} Personen:"
)
print(large_valid_clusters)


# ============================================================
# 8. Funktion für Standardisierung und Spearman-Vergleich
# ============================================================

def compare_cluster_profiles(
    disc_cluster_means,
    valid_cluster_means,
    usable_variables,
    discovery_mean,
    discovery_sd,
    title
):
    """
    Standardisiert Discovery- und Validation-Clusterprofile
    anhand der gewichteten Discovery-Gesamtmittelwerte und
    -standardabweichungen und berechnet anschließend die
    Spearman-Korrelationen zwischen allen Profilpaaren.
    """

    if disc_cluster_means.empty:
        raise ValueError(
            f"{title}: Keine Discovery-Cluster vorhanden."
        )

    if valid_cluster_means.empty:
        raise ValueError(
            f"{title}: Keine Validation-Cluster vorhanden."
        )

    # Clusterprofile anhand derselben Discovery-Skala standardisieren
    disc_profiles_z = (
        disc_cluster_means[usable_variables]
        - discovery_mean[usable_variables]
    ) / discovery_sd[usable_variables]

    valid_profiles_z = (
        valid_cluster_means[usable_variables]
        - discovery_mean[usable_variables]
    ) / discovery_sd[usable_variables]

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    print("\nStandardisierte Discovery-Clusterprofile:")
    display(disc_profiles_z.round(3))

    print("Standardisierte Validation-Clusterprofile:")
    display(valid_profiles_z.round(3))

    # Leere Ergebnismatrizen
    spearman_matrix = pd.DataFrame(
        index=[
            f"Discovery {cluster}"
            for cluster in disc_profiles_z.index
        ],
        columns=[
            f"Validation {cluster}"
            for cluster in valid_profiles_z.index
        ],
        dtype=float
    )

    p_value_matrix = spearman_matrix.copy()
    n_variables_matrix = spearman_matrix.copy()

    # Profilkorrelationen berechnen
    for discovery_cluster in disc_profiles_z.index:
        for validation_cluster in valid_profiles_z.index:

            discovery_profile = disc_profiles_z.loc[
                discovery_cluster
            ]

            validation_profile = valid_profiles_z.loc[
                validation_cluster
            ]

            # Nur Variablen verwenden, die in beiden Profilen gültig sind
            complete = (
                discovery_profile.notna()
                & validation_profile.notna()
            )

            x = discovery_profile.loc[complete]
            y = validation_profile.loc[complete]

            row_name = f"Discovery {discovery_cluster}"
            column_name = f"Validation {validation_cluster}"

            n_variables_matrix.loc[
                row_name,
                column_name
            ] = len(x)

            if (
                len(x) < 3
                or x.nunique() < 2
                or y.nunique() < 2
            ):
                rho = np.nan
                p_value = np.nan
            else:
                rho, p_value = spearmanr(x, y)

            spearman_matrix.loc[
                row_name,
                column_name
            ] = rho

            p_value_matrix.loc[
                row_name,
                column_name
            ] = p_value

    print("\nSpearman-Ähnlichkeitsmatrix der Clusterprofile:")
    display(spearman_matrix.round(3))

    print("p-Wert-Matrix:")
    display(p_value_matrix.round(4))

    print("Anzahl der jeweils verwendeten Variablen:")
    display(n_variables_matrix.astype("Int64"))

    return {
        "disc_profiles_z": disc_profiles_z,
        "valid_profiles_z": valid_profiles_z,
        "spearman_matrix": spearman_matrix,
        "p_value_matrix": p_value_matrix,
        "n_variables_matrix": n_variables_matrix
    }


# ============================================================
# 9. Vergleich 1: Alle Cluster
# ============================================================

results_all_clusters = compare_cluster_profiles(
    disc_cluster_means=disc_cluster_means_all,
    valid_cluster_means=valid_cluster_means_all,
    usable_variables=usable_variables,
    discovery_mean=discovery_mean,
    discovery_sd=discovery_sd,
    title="Profilvergleich: alle Cluster"
)


# ============================================================
# 10. Vergleich 2: Nur Cluster mit mindestens 100 Personen
# ============================================================

# Sicherstellen, dass die ausgewählten Cluster auch im
# Index der gespeicherten Summary-Tabellen vorhanden sind
large_disc_clusters_available = [
    cluster
    for cluster in large_disc_clusters
    if cluster in disc_cluster_means_all.index
]

large_valid_clusters_available = [
    cluster
    for cluster in large_valid_clusters
    if cluster in valid_cluster_means_all.index
]

missing_disc_clusters = sorted(
    set(large_disc_clusters)
    - set(large_disc_clusters_available)
)

missing_valid_clusters = sorted(
    set(large_valid_clusters)
    - set(large_valid_clusters_available)
)

if missing_disc_clusters:
    print(
        "Warnung: Folgende ausreichend große Discovery-Cluster "
        "fehlen in den Summary-Tabellen:"
    )
    print(missing_disc_clusters)

if missing_valid_clusters:
    print(
        "Warnung: Folgende ausreichend große Validation-Cluster "
        "fehlen in den Summary-Tabellen:"
    )
    print(missing_valid_clusters)

disc_cluster_means_100 = (
    disc_cluster_means_all
    .loc[large_disc_clusters_available]
    .copy()
)

valid_cluster_means_100 = (
    valid_cluster_means_all
    .loc[large_valid_clusters_available]
    .copy()
)

results_clusters_100 = compare_cluster_profiles(
    disc_cluster_means=disc_cluster_means_100,
    valid_cluster_means=valid_cluster_means_100,
    usable_variables=usable_variables,
    discovery_mean=discovery_mean,
    discovery_sd=discovery_sd,
    title=(
        f"Profilvergleich: nur Cluster mit mindestens "
        f"{min_cluster_size} Personen"
    )
)

Laut Metadaten ausgewählt: 39 kontinuierliche/ordinale Variablen
Gefundene Mittelwertspalten: Discovery=106, Validation=106

Erste Mittelwertspalten:
['zeit', 'ges', 'alter', 'altq', 'hh', 'hh14', 'allein', 'hne', 'vx210', 'wein30gr', 'mish30gr', 'bier30gr', 'audit', 'bierkons', 'weinkons', 'spirkons', 'mishkons', 'binge', 'alk30gr', 'alk30kat']
In beiden Cluster-Summaries vorhanden: 38 Variablen

Variablen ohne verwendbare Discovery-SD:


,Grund
audit,keine Varianz
fag_sch,keine Varianz
psecdi_sch,keine gültigen Werte mit positivem Gewicht
hnb_fag_sch,keine gültigen Werte mit positivem Gewicht
sds_c,keine gültigen Werte mit positivem Gewicht
sds_k,keine gültigen Werte mit positivem Gewicht
sds_a,keine gültigen Werte mit positivem Gewicht
fr12_meth1,keine Varianz
pomi,keine gültigen Werte mit positivem Gewicht



Verwendete Variablen: 29
Entfernte Variablen: 9

Clustergrößen Discovery:


,n
label,
0,711
1,263
2,3119
3,1
4,194
5,8
6,20
7,1526
8,65


Clustergrößen Validation:


...

,Validation 0,Validation 1,Validation 2,Validation 3,Validation 4,Validation 5,Validation 6,Validation 7,Validation 8,Validation 9,...,Validation 14,Validation 15,Validation 16,Validation 17,Validation 18,Validation 19,Validation 20,Validation 21,Validation 22,Validation 23
Discovery 0,0.0000,0.0006,0.2576,0.0000,0.0002,0.0003,0.0002,0.0000,0.0000,0.0000,...,0.0015,0.0000,0.0185,0.0000,0.0000,0.0000,0.0001,0.0005,0.1133,0.0000
Discovery 1,0.0000,0.0010,0.2062,0.0000,0.0000,0.0001,0.0000,0.0000,0.0000,0.0000,...,0.0015,0.0000,0.0096,0.0000,0.0000,0.0000,0.0000,0.0004,0.1640,0.0000
Discovery 2,0.0000,0.0000,0.4412,0.0000,0.0011,0.0026,0.0024,0.0001,0.0040,0.0023,...,0.0001,0.0000,0.0298,0.0000,0.0000,0.0051,0.0010,0.0029,0.7440,0.0001
Discovery 3,0.7860,0.1452,0.0078,0.7607,0.0231,0.0111,0.0547,0.1842,0.1738,0.3697,...,0.0771,0.5112,0.1959,0.5396,0.6058,0.0721,0.1824,0.0260,0.0310,0.2289
Discovery 4,0.0034,0.0000,0.0300,0.0009,0.0000,0.0000,0.0000,0.0001,0.0013,0.0014,...,0.0000,0.0012,0.0007,0.0004,0.0010,0.0000,0.0000,0.0000,0.1188,0.0000
Discovery 5,0.0000,0.0212,0.0568,0.0040,0.0000,0.0009,0.0000,0.0000,0.0000,0.0000,...,0.0035,0.0001,0.0004,0.0001,0.0000,0.0001,0.0001,0.0011,0.1100,0.0001
Discovery 6,0.0000,0.0001,0.0112,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.1413,0.0000
Discovery 7,0.0006,0.0000,0.3781,0.0000,0.0000,0.0000,0.0000,0.0001,0.0187,0.0279,...,0.0000,0.0000,0.0031,0.0003,0.0052,0.0001,0.0002,0.0000,0.7503,0.0000
Discovery 8,0.0001,0.0001,0.0071,0.0014,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0007,0.0001,0.0000,0.0000,0.0000,0.0000,0.0863,0.0000
Discovery 9,0.7879,0.2780,0.4024,0.9697,0.1089,0.0505,0.1968,0.4082,0.1337,0.2310,...,0.1170,0.9960,0.1941,0.5329,0.9575,0.1176,0.2446,0.0529,0.7651,0.2804


Anzahl der jeweils verwendeten Variablen:


,Validation 0,Validation 1,Validation 2,Validation 3,Validation 4,Validation 5,Validation 6,Validation 7,Validation 8,Validation 9,...,Validation 14,Validation 15,Validation 16,Validation 17,Validation 18,Validation 19,Validation 20,Validation 21,Validation 22,Validation 23
Discovery 0,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 1,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 2,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 3,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 4,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 5,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 6,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 7,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 8,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29
Discovery 9,29,29,29,29,29,29,29,29,29,29,...,29,29,29,29,29,29,29,29,28,29


Warnung: Folgende ausreichend große Discovery-Cluster fehlen in den Summary-Tabellen:
['0', '1', '12', '15', '16', '2', '21', '22', '4', '7']
Warnung: Folgende ausreichend große Validation-Cluster fehlen in den Summary-Tabellen:
['0', '1', '10', '11', '13', '14', '15', '17', '18', '3', '7', '8', '9']


ValueError: Profilvergleich: nur Cluster mit mindestens 100 Personen: Keine Discovery-Cluster vorhanden.